# Buổi 2, bài tập về nhà: incremental load order_items

order_items là bảng con của orders. Làm tương tự Bài tập 1 (`c2-practice_01`), nhưng bạn tự quyết định watermark. Bảng có cột `updated_at`, vậy khi một đơn đổi trạng thái thì order_items của đơn đó có đổi `updated_at` không? Kiểm tra trên data thật rồi mới chọn.

Phần 1 là code: chọn watermark, ghi append-only vào Raw Zone, backfill 7 ngày gần nhất bằng tham số ngày. Phần 2 là README, trả lời 3 câu ở cuối notebook.

Trao đổi thêm ở Facebook Group lớp học, nộp bài qua Hệ thống học tập.

Import thư viện, khai báo cấu hình, tạo client GCS và kết nối database, giống Bài tập 1. File `.env` và key GCP dùng lại từ buổi học.

In [31]:
import glob
import os
import uuid
from datetime import date, datetime, timedelta, timezone

import pandas as pd
import sqlalchemy as sa
from dotenv import load_dotenv
from google.cloud import storage
from google.oauth2 import service_account

load_dotenv("../../secrets/.env", override=True)

True

In [32]:
GCP_PROJECT = os.environ.get("GCP_PROJECT")
DEST_BUCKET = os.environ.get("DEST_BUCKET")
GCP_LOCATION = os.environ.get("GCP_LOCATION", "asia-southeast1")
GCP_KEY_PATH = os.environ.get("GCP_KEY_PATH", "../../secrets/hien-nguyen-cybersoft-e70ce34d9f1f.json")
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT", 6543)
DB_USER = os.environ.get("DB_USER")
DB_PWD = os.environ.get("DB_PWD")
DB_NAME = os.environ.get("DB_NAME", "postgres")

LOCAL_PATH = "../../output/raw"

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PWD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

In [33]:
credentials = service_account.Credentials.from_service_account_file(GCP_KEY_PATH)
gcs_client = storage.Client(project=GCP_PROJECT, credentials=credentials)

bucket = gcs_client.bucket(DEST_BUCKET)
if not bucket.exists():
    bucket = gcs_client.create_bucket(DEST_BUCKET, location=GCP_LOCATION)
    print(f"Created bucket: gs://{DEST_BUCKET}")
else:
    print(f"Bucket exists: gs://{DEST_BUCKET}")

Bucket exists: gs://de01_laurenprj123


In [34]:
engine = sa.create_engine(DB_URL)

Hai function dùng lại từ Bài tập 1: `write_raw` ghi local rồi upload lên bucket, `read_day` đọc lại một ngày từ bản local cho bước kiểm tra cuối bài. Giữ nguyên.

In [36]:
print(LOCAL_PATH);
def write_raw(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)  # ghi local trước, phòng upload lỗi giữa chừng
    bucket.blob(path.removeprefix("./")).upload_from_filename(path)  # rồi đẩy lên GCS cùng đường dẫn


def read_day(table, data_date):
    files = glob.glob(f"{LOCAL_PATH}/stylehub/{table}/date={data_date}/**/*.parquet", recursive=True)
    return pd.concat(pd.read_parquet(f) for f in files)

../../output/raw


In [35]:
print(DB_USER, DB_HOST, DB_PORT, DB_NAME)

student_readonly db.stylehub.delabs.io.vn 5432 stylehub


Bước 1: xem data order_items. Viết thêm query để trả lời câu hỏi: khi đơn đổi trạng thái, `updated_at` của order_items có đổi không? Gợi ý: so `updated_at` của order_items với `updated_at` của orders cho cùng `order_id`.

In [39]:
sql_check = """
         WITH a AS (
    SELECT
        o.id as order_id,
        oi.id AS item_id,
        o.status,
        oi.created_at AS item_created_at,
        o.updated_at  AS order_updated_at,
        oi.updated_at AS item_updated_at,
        EXTRACT(EPOCH FROM (o.updated_at - oi.updated_at)) AS lag_behind_sec
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.id
)
SELECT *
FROM a
WHERE lag_behind_sec > 0 OR item_updated_at IS NULL
ORDER BY lag_behind_sec desc
LIMIT 100;"""
df_check = pd.read_sql(sql_check, engine)
df_check.head(10)

# TODO: viết query kiểm tra câu hỏi ở trên

,order_id,item_id,status,item_created_at,order_updated_at,item_updated_at,lag_behind_sec
0,107215,155097,Returned,2026-09-30 22:02:30.576668,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
1,92243,133382,Returned,2026-09-30 21:41:48.437877,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
2,117872,170447,Returned,2026-09-30 22:43:19.893974,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
3,76686,110846,Returned,2026-09-30 22:55:43.128505,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
4,118943,171983,Returned,2026-09-30 19:33:45.932908,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
5,92615,133926,Returned,2026-09-30 16:06:20.218097,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
6,65715,94876,Returned,2026-09-30 19:17:13.303313,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
7,72272,104436,Returned,2026-09-30 22:13:00.054308,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
8,79972,115599,Returned,2026-09-30 23:22:10.257558,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303
9,80415,116240,Returned,2026-09-30 16:00:15.002907,2026-10-07 15:22:38.113354,2026-10-01 14:31:29.907051,521468.206303


Bước 2: chọn cột watermark. Lý do viết ở phần README cuối bài.

In [41]:
WATERMARK_COL = "updated_at"  # TODO: created_at hay updated_at

Bước 3: lấy data của một ngày và ghi vào Raw Zone, append-only (mỗi lần chạy một file mới trong folder có `batch_id`). Đường dẫn: `LOCAL_PATH/stylehub/order_items/date=<data_date>/batch_id=<batch_id>/order_items.parquet`.

In [67]:
from psycopg2 import Timestamp
data_date = "2026-10-01"
# or:
# data_date = pd.Timestamp("2026-09-30").strftime("%Y-%m-%d")

# TODO 1: query order_items có WATERMARK_COL nằm trong data_date và data_date + 1 ngày
df = pd.read_sql(
    sa.text(f"""
        SELECT *
        FROM public.order_items
        WHERE {WATERMARK_COL} >= CAST(:data_date AS date)
          AND {WATERMARK_COL} < CAST(:data_date AS date) + interval '1 day'
    """),
    engine,
    params={"data_date": data_date},
)
df.head(10)

# TODO 2: thêm 3 cột metadata _ingested_at, _source, _batch_id
print(LOCAL_PATH)
batch_id = uuid.uuid4().hex[:12]
df = df.assign(_ingested_at=datetime.now(timezone.utc), _source="db:public.order_items", _batch_id=batch_id)
#Kiểm tra số lượng dòng theo ngày để đảm bảo không bị thiếu dữ liệu
#select updated_at::date, count(*)
#from order_items
#group by updated_at::date
#order by updated_at::date

# TODO 3: ghi vào Raw Zone theo đường dẫn ở trên bằng write_raw
path = f"{LOCAL_PATH}/stylehub/order_items/date={data_date}/batch_id={batch_id}/order_items.parquet"
write_raw(df, path)
print(data_date, "lấy", len(df), "dòng ->", path)

../../output/raw
2026-10-01 lấy 183260 dòng -> ../../output/raw/stylehub/order_items/date=2026-10-01/batch_id=5cc229201a5e/order_items.parquet


Bước 4: backfill 7 ngày gần nhất, rồi chạy cả vòng thêm một lần nữa. Sau mỗi vòng, đọc lại bằng `read_day` và lưu số id khác nhau của từng ngày vào `ids_per_run`. Dùng lại phần code ở Bước 3, đặt trong vòng lặp qua `DAYS`.

In [68]:
data_date="2026-10-01"
print(data_date)
day_back = lambda n: (date.fromisoformat(data_date) - timedelta(days=n)).isoformat() 
DAYS = [day_back(i) for i in range(7, 0, -1)]
# TODO: 7 ngày gần nhất, định dạng YYYY-MM-DD
print(DAYS)
ids_per_run = []

# TODO: for run in [1, 2], bên trong lặp qua DAYS, ingest như Bước 3, rồi đọc lại bằng read_day
for run in [1, 2]:
    for data_date in DAYS:
        #print("run_date:", run_date, "| data_date:", data_date)
        df = pd.read_sql(
            sa.text(f"""
            SELECT *
            FROM public.order_items
            WHERE {WATERMARK_COL} >= CAST(:data_date AS date)
            AND {WATERMARK_COL} < CAST(:data_date AS date) + interval '1 day'
            """),
            engine,
            params={"data_date": data_date},
        )

        batch_id = uuid.uuid4().hex[:12]
        df = df.assign(_ingested_at=datetime.now(timezone.utc), _source="db:public.order_items", _batch_id=batch_id)
        path = f"{LOCAL_PATH}/stylehub/order_items/date={data_date}/batch_id={batch_id}/order_items.parquet"
        write_raw(df, path)

    days = {d: read_day("order_items", d) for d in DAYS}
    ids = {d: x["id"].nunique() for d, x in days.items()}
    print("lần", run, "| số dòng:", {d: len(x) for d, x in days.items()}, "| số id khác nhau:", ids)
    ids_per_run.append(ids)

2026-10-01
['2026-09-24', '2026-09-25', '2026-09-26', '2026-09-27', '2026-09-28', '2026-09-29', '2026-09-30']
lần 1 | số dòng: {'2026-09-24': 0, '2026-09-25': 0, '2026-09-26': 0, '2026-09-27': 0, '2026-09-28': 0, '2026-09-29': 0, '2026-09-30': 0} | số id khác nhau: {'2026-09-24': 0, '2026-09-25': 0, '2026-09-26': 0, '2026-09-27': 0, '2026-09-28': 0, '2026-09-29': 0, '2026-09-30': 0}
lần 2 | số dòng: {'2026-09-24': 0, '2026-09-25': 0, '2026-09-26': 0, '2026-09-27': 0, '2026-09-28': 0, '2026-09-29': 0, '2026-09-30': 0} | số id khác nhau: {'2026-09-24': 0, '2026-09-25': 0, '2026-09-26': 0, '2026-09-27': 0, '2026-09-28': 0, '2026-09-29': 0, '2026-09-30': 0}


Bước 5: kiểm tra chạy lại không làm đổi số id khác nhau. Kết quả đúng là True.

In [66]:
print("số id khác nhau không đổi:", ids_per_run[0] == ids_per_run[1])

số id khác nhau không đổi: True


## README

Trả lời 3 câu ngay dưới đây.

1. Vì sao chọn cột watermark đó, lookback bao nhiêu ngày?
2. Chạy lại thì Raw Zone có gì, bỏ dòng trùng bằng cách nào?
3. Nguồn xóa một dòng order_items thì pipeline có biết không?

Trả lời:
1. Chọn update_at, vì created_at tại 2 bảng không bắt được dòng đã thay đổi trạng thái, còn updated_at thì bắt được, nhưng lưu ý Output ở bước 1 cho thấy tbl Orders sau khi thay đổi Status thì Updated_at thay đổi nhưng các dòng items của tbl Order_items không update Updated_at nên Update_at của 2 bảng có độ lệch.
Theo output dữ liệu thì cột lag_behind_sec giữa 2 bảng ~6 ngày, nên đặt lookback 7 ngày = backfill 7 ngày

2. Mỗi lần chạy thì Raw zone giữ 1 phiên bản khác, không overwrite do khác batchid và _ingested_at=datetime.now(timezone.utc); có thể loại bỏ dòng trùng trên tầng staging, nhóm theo id, giữ dòng có _ingested_at mới nhất (drop_duplicates hoặc row_number() over (partition by id order by _ingested_at desc) = 1)

3. Nguồn xóa 1 dòng order_items thì pipeline không biết vì nó chỉ lưu các dòng có _updated_at thay đổi (tức là chỉ Append only). Muốn phát hiện các dòng xóa cần soft delete (cột deleted_at), hoặc so vs snapshot toàn bảng định kỳ để flag các dòng bị deleted